In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# 1. Define two versions of the same CNN
class MNISTNet(nn.Module):
    def __init__(self, use_bn=False):
        super(MNISTNet, self).__init__()
        layers = [
            nn.Conv2d(1, 10, kernel_size=5),
            nn.ReLU()
        ]
        if use_bn:
            layers.append(nn.BatchNorm2d(10)) # The Magic Layer
        
        layers.append(nn.MaxPool2d(2))
        layers.extend([
            nn.Flatten(),
            nn.Linear(10 * 12 * 12, 10)
        ])
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

# 2. Setup Data
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('./data', train=True, download=True, transform=transform), 
    batch_size=64, shuffle=True
)

# 3. Training Function
def train_and_get_history(use_bn):
    model = MNISTNet(use_bn=use_bn).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.1) # High LR to test stability
    criterion = nn.CrossEntropyLoss()
    history = []
    
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        if batch_idx > 100: break # Only train 100 batches for quick comparison
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        # Calculate Accuracy
        pred = output.argmax(dim=1, keepdim=True)
        acc = pred.eq(target.view_as(pred)).sum().item() / len(data)
        history.append(acc)
    return history

# 4. Run Experiment
print("Training Model WITHOUT BatchNorm...")
history_no_bn = train_and_get_history(use_bn=False)

print("Training Model WITH BatchNorm...")
history_with_bn = train_and_get_history(use_bn=True)

# 5. Plot Results
plt.figure(figsize=(10, 6))
plt.plot(history_no_bn, label='Without BatchNorm', alpha=0.7)
plt.plot(history_with_bn, label='With BatchNorm', lw=2)
plt.title("Effect of BatchNorm on MNIST Training Speed")
plt.xlabel("Batches")
plt.ylabel("Training Accuracy")
plt.legend()
plt.grid(True)
plt.show()